# 30 · Scale-Up Experiments (Reviewer Fixes #7–9)

Addresses the three "needs modest additional work" items from review feedback:

- **#7** — TMR real-channel trials: `n=20 -> n=100` (tighter Wilson CI on the 0%-correction finding)
- **#8** — A second FABRIC site pair with different latency/congestion, to test whether "real network doesn't matter for silent divergence" holds beyond one low-noise link
- **#9** — 5 more real key pairs (5 -> 10), to de-risk Table 2 (finite-key length) from being sensitive to any one key

Each section is self-contained — run whichever you need. Sections B and C provision/reuse a
live FABRIC slice and will take a while (network round-trips); start them and let them run.

**Prerequisite:** existing repo checked out with `results/` populated from your original runs
(this notebook reads several of your existing CSVs to append to / compare against them).

## Section A — TMR: extend n=20 → n=100 (Feedback #7)

Reuses the exact `tmr_real_channel_trial` mechanism from
`notebooks/mitigation_v2_redundant_pa_tmr.ipynb`, at the same `fault_prob=0.3`, same key
(key0), just more trials. Appends to your existing `results/mitigation_tmr_realchannel.csv`
rather than overwriting it, so the first 20 trials you already collected are preserved.

In [1]:
import sys, json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.key import key_from_sifted_json

RESULTS = PROJECT_DIR / "results"
print("Imports complete")


Imports complete


In [2]:
# --- Connect to your existing slice and re-sync key0 (same as mitigation_v2 notebook) ---
SLICE_NAME = 'qfabric-bb84-2'

fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

GEN_ALICE = RESULTS / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = RESULTS / "fabric_bob_sifted_bits_genonly.json"
META_PATH = RESULTS / "fabric_key0_pe_meta.json"

alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
assert alice_indices == bob_indices

meta = json.loads(META_PATH.read_text())
k_pe, real_qber = meta["k"], meta["qber"]
print(f"key0 loaded: n={alice_key.get_nr_bits()} bits, k={k_pe}, qber={real_qber:.4f}")

alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")
print("Remote nodes re-synced to key0")


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


SliceNotFoundError: 'Unable to find slice "qfabric-bb84-2" for this project. Check slice name spelling and project id.'

In [ ]:
def extract_output_path(stdout_text, marker):
    match = re.search(rf"{marker} (\S+\.json)", stdout_text)
    return match.group(1) if match else None


def tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run, base_seed, fault_prob, k,
                            n_replicas=3, timeout=180):
    """Unchanged from mitigation_v2_redundant_pa_tmr.ipynb -- copied here so this
    notebook doesn't depend on that one having been run first in the same kernel."""
    replica_results = []
    for rep in range(n_replicas):
        seed = base_seed + rep * 1000
        bob_output = f"results/bob_tmr_run{run}_rep{rep}.json"

        bob_thread = bob.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/bob_cascade_driver.py "
            f"--key-json results/bob_sifted_bits.json --alice-key-json results/alice_sifted_bits.json "
            f"--host {bob_ip} --port 5200 --qber {real_qber} --k {k} --seed {seed} "
            f"--reconciliation-prob {fault_prob} --output {bob_output}"
        )
        alice_thread = alice.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/alice_cascade_responder.py "
            f"--key-json results/alice_sifted_bits.json --bob-host {bob_ip} --port 5200 "
            f"--seed {seed} --output results/alice_tmr_run{run}_rep{rep}.json"
        )

        bob_out = bob_thread.result(timeout=timeout)
        try:
            alice_thread.result(timeout=timeout // 2)
        except Exception:
            pass

        bob_output_path = extract_output_path(bob_out[0], "Result written to")
        if bob_output_path is None:
            replica_results.append({"status": "crashed", "bits": None})
            continue

        stdout, _ = bob.execute(f"cat ~/qfabric/{bob_output_path}", quiet=True)
        try:
            data = json.loads(stdout)
        except json.JSONDecodeError:
            replica_results.append({"status": "no_output", "bits": None})
            continue

        if data.get("non_convergent"):
            replica_results.append({"status": "non_convergent", "bits": None})
        elif "bob_reconciled_bits" in data:
            replica_results.append({
                "status": "converged", "bits": np.array(data["bob_reconciled_bits"]),
                "remaining_errors": data.get("remaining_errors_after_reconciliation"),
            })
        else:
            replica_results.append({"status": "converged_no_bits_saved", "bits": None})

        time.sleep(2)
    return replica_results

print("tmr_real_channel_trial defined")


In [ ]:
# --- Run 80 more trials (indices 20..99), continuing seeds from where the
# original 20 left off, and append to the SAME csv the paper's Table/Fig 9 was built from ---
TMR_CSV = RESULTS / "mitigation_tmr_realchannel.csv"
existing_tmr = pd.read_csv(str(TMR_CSV))
start_run = int(existing_tmr["run"].max()) + 1  # continues at 20 if you have the original 20
N_NEW_TRIALS = 80  # -> 100 total; raise/lower as time allows, CI tightens with more

new_rows = []
for run in range(start_run, start_run + N_NEW_TRIALS):
    base_seed = 42 + run * 10
    replicas = tmr_real_channel_trial(bob, alice, bob_ip, real_qber, run, base_seed,
                                       fault_prob=0.3, k=k_pe)
    statuses = [r["status"] for r in replicas]
    valid = [r["bits"] for r in replicas if r["bits"] is not None]

    if len(valid) < 2:
        new_rows.append({"run": run, "statuses": statuses, "result": "insufficient_valid_replicas",
                          "n_valid": len(valid), "corrected": False})
        print(f"run {run}: insufficient valid replicas ({statuses})")
        continue

    majority = (np.stack(valid).sum(axis=0) > len(valid) / 2).astype(np.uint8)
    n_errors = alice_key.nr_bits_different(Key(bits=majority))
    corrected = n_errors == 0
    new_rows.append({"run": run, "statuses": statuses, "result": "voted",
                      "n_valid": len(valid), "n_errors_after_tmr": n_errors, "corrected": corrected})
    print(f"run {run}: n_valid={len(valid)}, errors_after_vote={n_errors}, corrected={corrected}")

    # Checkpoint after every trial -- real-channel runs are slow and this can be
    # interrupted/resumed safely since start_run is read from the file each time.
    df_checkpoint = pd.concat([existing_tmr, pd.DataFrame(new_rows)], ignore_index=True)
    df_checkpoint.to_csv(str(TMR_CSV), index=False)

print(f"\nDone. Total trials in {TMR_CSV.name}: {len(df_checkpoint)}")


In [ ]:
# --- Recompute the Wilson 95% CI on the correction rate at n=100 (replaces the
# n=20, [0%, 16.1%] CI reported in the submitted draft) ---
from scipy.stats import norm

def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return np.nan, np.nan, np.nan
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    halfwidth = (z / denom) * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    return p_hat, max(0, center - halfwidth), min(1, center + halfwidth)

df_final = pd.read_csv(str(TMR_CSV))
voted = df_final[df_final["result"] == "voted"]
n_trials = len(voted)
n_corrected = int(voted["corrected"].sum())
p_hat, lo, hi = wilson_ci(n_corrected, n_trials)

mean_errors = voted["n_errors_after_tmr"].mean()
se_errors = voted["n_errors_after_tmr"].std() / np.sqrt(n_trials)

print(f"n={n_trials} real-channel TMR trials")
print(f"corrected: {n_corrected}/{n_trials}  (p_hat={p_hat:.3f})")
print(f"95% Wilson CI on true correction rate: [{lo*100:.1f}%, {hi*100:.1f}%]")
print(f"mean residual errors after vote: {mean_errors:.2f} +/- {se_errors:.2f} (SE)")


## Section B — Second FABRIC site pair (Feedback #8)

Your original pair was **NCSA <-> EDUKY** (Illinois <-> Kentucky): geographically close,
deliberately picked as "clear, robust, minimal noise." To test whether the mock-vs-real
non-effect (Section 4.3 / CMH p=0.16, p=0.32) generalizes, or was itself a property of that
particular low-noise link, we want a pair with **more distance and more hops**.

FABRIC's site list (per the project's public site info) includes coast-adjacent sites such
as `UTAH`, `MASS` (MGHPCC), `FIU`, `UCSD`, `MICH`, and `CLEM`, alongside the ESnet backbone
production sites (`STAR`/Starlight, `SALT`, `DALL`). Check live capacity before committing —
site availability on FABRIC fluctuates — with `fablib.list_sites()`.

**Suggested pair:** keep Alice + the BMv2 switch at `NCSA` (so Section A's key/switch config
carries over) and move Bob to `UTAH` — a genuinely different coast, more router hops, and a
different jitter profile than the NCSA-EDUKY link. Swap `SITE_BOB2` for whatever
`list_sites()` shows has capacity if `UTAH` doesn't.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.config import ScenarioConfig
from qne.cascade.sweep_utils import run_real_channel_trial, collect_key_pairs

RESULTS = PROJECT_DIR / "results"
fablib = deploy.get_fablib()

# --- Sanity-check capacity before provisioning; adjust SITE_BOB2 based on what's free ---
for site in ["TOKY", "UTAH", "FIU", "UCSD", "MICH", "CLEM", "STAR", "SALT"]:
    try:
        print(site, fablib.get_site_attribute(site, "cores_available"))
    except Exception as e:
        print(site, "unavailable:", e)


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


TOKY unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
UTAH unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
FIU unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
UCSD unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
MICH unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
CLEM unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
STAR unavailable: 'FablibManager' object has no attribute 'get_site_attribute'
SALT unavailable: 'FablibManager' object has no attribute 'get_site_attribute'


In [3]:
SLICE_NAME_2 = 'qfabric-bb84-site2'
SITE_ALICE_2 = 'NCSA'
SITE_BOB_2   = 'TOKY'   # <- change based on the capacity check above if needed
SITE_SW_2    = 'NCSA'
SCENARIO     = 'validation/scenarios/fabric_1km.yml'
BMV2_IMAGE   = 'ghcr.io/kthare10/qfabric-bmv2:latest'

try:
    slice_obj_2 = fablib.get_slice(name=SLICE_NAME_2)
    print(f"Reusing existing slice '{SLICE_NAME_2}' (state: {slice_obj_2.get_state()})")
except Exception:
    slice_obj_2 = deploy.create_slice(fablib, SLICE_NAME_2, SITE_ALICE_2, SITE_BOB_2, SITE_SW_2)

deploy.upload_project(slice_obj_2)
import os
os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE
deploy.install_deps(slice_obj_2, build_bmv2=False)
deploy.setup_switch_docker(slice_obj_2, BMV2_IMAGE)

threshold = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO).loss_threshold_u32
alice_mac_2, bob_mac_2, sw_alice_mac_2, sw_bob_mac_2, _, _ = deploy.configure_switch(slice_obj_2, threshold)
alice_ip_2, bob_ip_2 = deploy.setup_dataplane_ips(slice_obj_2, alice_mac_2, bob_mac_2)

alice_2 = slice_obj_2.get_node("alice")
bob_2 = slice_obj_2.get_node("bob")
print(f"\nSite-2 slice ready: {SITE_ALICE_2} <-> {SITE_BOB_2}, switch at {SITE_SW_2}")


Reusing existing slice 'qfabric-bb84-site2' (state: Configuring)

=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...


ValidationError: Invalid management IP: None

In [5]:
try:
    slice_obj_2 = fablib.get_slice(name=SLICE_NAME_2)
    state = slice_obj_2.get_state()
    print(f"Found existing slice '{SLICE_NAME_2}', state: {state}")
    if state not in ("StableOK", "Active"):  # adjust if your fablib version reports differently
        raise RuntimeError(f"Slice exists but isn't active (state={state}) -- treating as expired")
except Exception as e:
    print(f"Existing slice unusable ({e}), creating fresh...")
    try:
        # Clean up the dead record first so the name is free
        slice_obj_2.delete()
    except Exception:
        pass
    slice_obj_2 = deploy.create_slice(fablib, SLICE_NAME_2, SITE_ALICE_2, SITE_BOB_2, SITE_SW_2)

deploy.upload_project(slice_obj_2)

Found existing slice 'qfabric-bb84-site2', state: StableOK

=== Uploading project (clean tarball) ===
  Uploading to alice...
  Uploading to bob...
  Uploading to switch...
  Upload complete (qne + validation + scenarios + p4 on every node)


In [ ]:
import json

fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name='qfabric-bb84-2')
bob = slice_obj.get_node("bob")

known_metadata = [
    {"index": 0, "m_sifted": 3894, "k_pe": 390, "n_bits": 3504, "qber": 0.01282051282051282},
    {"index": 1, "m_sifted": 3788, "k_pe": 379, "n_bits": 3409, "qber": 0.0158311345646438},
    {"index": 2, "m_sifted": 3734, "k_pe": 374, "n_bits": 3360, "qber": 0.00267379679144385},
    {"index": 3, "m_sifted": 3813, "k_pe": 382, "n_bits": 3431, "qber": 0.002617801047120419},
    {"index": 4, "m_sifted": 3841, "k_pe": 385, "n_bits": 3456, "qber": 0.005194805194805195},
]
for row in known_metadata:
    i = row["index"]
    for who, remote_name in [("alice", f"alice_sifted_bits_key{i}.json"), ("bob", f"bob_sifted_bits_key{i}.json")]:
        local_path = RESULTS / remote_name
        stdout, _ = bob.execute(f"cat ~/qfabric/results/{remote_name}", quiet=True)
        json.loads(stdout)  # sanity check
        local_path.write_text(stdout)
    row["alice_path"] = str(RESULTS / f"alice_sifted_bits_key{i}.json")
    row["bob_path"] = str(RESULTS / f"bob_sifted_bits_key{i}.json")
    row["alice_raw_path"] = None
    row["bob_raw_path"] = None
print("recovered keys 0-4")

In [8]:
from qne.cascade.key import key_from_sifted_json
from qne.cascade.parameter_estimation import split_pe_and_generation

recovered_5_to_9 = []
for i in range(5, 10):
    alice_raw = RESULTS / f"alice_sifted_bits_key{i}_raw.json"
    bob_raw = RESULTS / f"bob_sifted_bits_key{i}_raw.json"
    if not alice_raw.exists():
        print(f"key {i}: _raw file missing, can't recompute -- check diagnostic output")
        continue
    alice_key_full, alice_idx = key_from_sifted_json(str(alice_raw), "alice_bits")
    bob_key_full, bob_idx = key_from_sifted_json(str(bob_raw), "bob_bits")
    split = split_pe_and_generation(alice_key_full, bob_key_full, sample_fraction=0.1, seed=100000 + i)
    recovered_5_to_9.append({"index": i, "m_sifted": split["m"], "k_pe": split["k"], "n_bits": split["n"],
                              "qber": split["qber"], "alice_path": str(RESULTS / f"alice_sifted_bits_key{i}.json"),
                              "bob_path": str(RESULTS / f"bob_sifted_bits_key{i}.json")})
    print(f"key {i}: recomputed m={split['m']}, k={split['k']}, n={split['n']}, qber={split['qber']:.5f}")

key 5: recomputed m=3839, k=384, n=3455, qber=0.01042
key 6: recomputed m=3829, k=383, n=3446, qber=0.02350
key 7: recomputed m=3765, k=377, n=3388, qber=0.01061
key 8: recomputed m=3812, k=382, n=3430, qber=0.01047
key 9: recomputed m=3724, k=373, n=3351, qber=0.01340


In [9]:
import pandas as pd
combined = pd.DataFrame(known_metadata + recovered_5_to_9)
combined.to_csv(str(RESULTS / "key_pairs_metadata.csv"), index=False)
print(f"key_pairs_metadata.csv rebuilt with {len(combined)} keys")

key_pairs_metadata.csv rebuilt with 10 keys


In [8]:
import sys
from pathlib import Path
import pandas as pd

# --- Diagnose cwd/PROJECT_DIR first -- this is almost certainly the actual issue ---
print("cwd:", Path.cwd())
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print("PROJECT_DIR resolved to:", PROJECT_DIR)
RESULTS = PROJECT_DIR / "results"
print("RESULTS resolved to:", RESULTS, "  exists:", RESULTS.exists())

sys.path.insert(0, str(PROJECT_DIR))
from qne.cascade.key import key_from_sifted_json


def load_key_pairs(metadata_csv):
    """Reload already-collected key pairs from disk -- no FABRIC slice needed.
    Accepts a Path OR a bare filename; if given a bare filename (or a path
    that doesn't exist as given), searches a few likely locations relative
    to PROJECT_DIR before giving up, and reports every path it tried."""
    metadata_csv = Path(metadata_csv)
    candidates = [
        metadata_csv,                                   # exactly what was passed
        RESULTS / metadata_csv.name,                     # results/<name>, in case a bare
                                                           # filename (no dir) was passed
        Path.cwd() / metadata_csv.name,                   # current working dir
        PROJECT_DIR / metadata_csv.name,                  # project root itself
    ]
    resolved = next((c for c in candidates if c.exists()), None)
    if resolved is None:
        tried = "\n  ".join(str(c.resolve()) for c in candidates)
        raise FileNotFoundError(
            f"Could not find '{metadata_csv.name}'. Tried:\n  {tried}\n"
            f"Run the key-collection cell again if this file was never written, "
            f"or pass the correct path explicitly."
        )
    print(f"loading {resolved}")

    df = pd.read_csv(str(resolved))
    keys = {}
    for _, row in df.iterrows():
        i = int(row["index"])
        # alice_path/bob_path in the CSV are absolute paths from when the CSV was
        # written -- if the results/ folder moved since, rebase to RESULTS instead.
        a_path = Path(row["alice_path"])
        b_path = Path(row["bob_path"])
        if not a_path.exists():
            a_path = RESULTS / a_path.name
        if not b_path.exists():
            b_path = RESULTS / b_path.name

        alice_key, alice_idx = key_from_sifted_json(str(a_path), "alice_bits")
        bob_key, bob_idx = key_from_sifted_json(str(b_path), "bob_bits")
        assert alice_idx == bob_idx, f"key {i}: matching_indices mismatch between alice/bob files"
        keys[i] = {"alice_key": alice_key, "bob_key": bob_key,
                   "k_pe": int(row["k_pe"]), "qber": float(row["qber"]), "n_bits": int(row["n_bits"])}
    return df, keys


# --- Site 1 (original + Section C's 5 extra keys, indices 0-9) ---
key_pairs_df, key_pairs = load_key_pairs("key_pairs_metadata.csv")
print(f"site 1: loaded {len(key_pairs)} keys, indices {sorted(key_pairs.keys())}")

# --- Site 2 (NCSA<->UTAH, indices 0-4) ---
key_pairs_df_2, key_pairs_2 = load_key_pairs("key_pairs_metadata_site2.csv")
print(f"site 2: loaded {len(key_pairs_2)} keys, indices {sorted(key_pairs_2.keys())}")a

cwd: /home/fabric/work/qkd-dependability/notebooks
PROJECT_DIR resolved to: /home/fabric/work/qkd-dependability
RESULTS resolved to: /home/fabric/work/qkd-dependability/results   exists: True


FileNotFoundError: Could not find 'key_pairs_metadata.csv'. Tried:
  /home/fabric/work/qkd-dependability/notebooks/key_pairs_metadata.csv
  /home/fabric/work/qkd-dependability/results/key_pairs_metadata.csv
  /home/fabric/work/qkd-dependability/notebooks/key_pairs_metadata.csv
  /home/fabric/work/qkd-dependability/key_pairs_metadata.csv
Run the key-collection cell again if this file was never written, or pass the correct path explicitly.

In [7]:
CSV_SITE2 = RESULTS / "sdc_site2_toeplitz_finalkey_doseresponse.csv"


In [5]:
# --- Same dose-response sweep as your paper's real-channel data (Section 3.5 /
# results/sdc_real_channel_toeplitz_finalkey_doseresponse_multikey_asymptotic.csv),
# run on the site-2 link instead. Lower N_RUNS_PER_PROB from the paper's n=150/n=500
# to start -- each point is a live FABRIC round trip; raise it once you've confirmed
# the pipeline runs cleanly end-to-end.
#
# NOTE: we upload each key under its OWN remote filename (alice/bob_sifted_bits_key{i}.json)
# and pass those paths straight to run_real_channel_trial via bob_key_json/alice_key_json,
# instead of re-uploading into the generic "results/alice_sifted_bits.json" name every
# loop iteration. That generic file is also written natively on the node by
# deploy.run_bb84() during collect_key_pairs_from above, and can end up owned/permissioned
# such that a later SFTP overwrite from this session gets a PermissionError (errno 13) --
# per-key filenames sidestep the clash entirely, and match how collect_key_pairs already
# names files on the remote side. ---
PROBS = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
N_RUNS_PER_PROB = 20  # paper used 150 for toeplitz/final-key; raise as time allows
CSV_SITE2 = RESULTS / "sdc_site2_toeplitz_finalkey_doseresponse.csv"

all_rows = []
for key_idx in range(5):
    meta_row = key_pairs_df_2.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])

    a_file = RESULTS / f"alice_sifted_bits_key{key_idx}_site2.json"
    b_file = RESULTS / f"bob_sifted_bits_key{key_idx}_site2.json"

    a_remote = f"results/alice_sifted_bits_key{key_idx}_site2.json"
    b_remote = f"results/bob_sifted_bits_key{key_idx}_site2.json"
    alice_2.upload_file(str(a_file), f"qfabric/{a_remote}")
    bob_2.upload_file(str(a_file), f"qfabric/{a_remote}")
    bob_2.upload_file(str(b_file), f"qfabric/{b_remote}")
    print(f"\n=== site2 key{key_idx}: qber={qber_i:.4f} ===")

    for prob in PROBS:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}),
                                    ("final_key", {"final_key_prob": prob})]:
            for run in range(N_RUNS_PER_PROB):
                seed = 42 + run
                result = run_real_channel_trial(bob_2, alice_2, bob_ip_2, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic",
                                                   bob_key_json=b_remote, alice_key_json=a_remote,
                                                   **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)
            # checkpoint per (key, prob, fault_type) block
            pd.DataFrame(all_rows).to_csv(str(CSV_SITE2), index=False)

print(f"\nSaved {len(all_rows)} rows -> {CSV_SITE2}")


NameError: name 'key_pairs_df_2' is not defined

In [1]:
# --- Step 1: diagnose. Changes nothing. ---
import sys, json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
RESULTS = PROJECT_DIR / "results"

import deploy_fabric as deploy
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name='qfabric-bb84-2')   # your ORIGINAL site-1 slice
bob = slice_obj.get_node("bob")

print("=== local results/ state ===")
for i in range(10):
    for who in ["alice", "bob"]:
        p = RESULTS / f"{who}_sifted_bits_key{i}.json"
        print(f"  {p.name}: {'present' if p.exists() else 'MISSING'}")
print("  key_pairs_metadata.csv:", (RESULTS / "key_pairs_metadata.csv").exists())

print("\n=== remote (bob, site-1 slice) state ===")
for i in range(10):
    stdout, _ = bob.execute(f"test -f ~/qfabric/results/alice_sifted_bits_key{i}.json && echo present || echo MISSING", quiet=True)
    print(f"  key{i} (alice file on bob node): {stdout.strip()}")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
=== local results/ state ===
  alice_sifted_bits_key0.json: MISSING
  bob_sifted_bits_key0.json: MISSING
  alice_sifted_bits_key1.json: MISSING
  bob_sifted_bits_key1.json: MISSING
  alice_sifted_bits_key2.json: MISSING
  bob_sifted_bits_key2.json: MISSING
  alice_sifted_bits_key3.json: MISSING
  bob_sifted_bits_key3.json: MISSING
  alice_sifted_bits_key4.json: MISSING
  bob_sifted_bits_key4.json: MISSING
  alice_sifted_bits_key5.json: MISSING
  bob_sifted_bits_key5.json: MISSING
  alice_sifted_bits_key6.json: MISSING
  bob_sifted_bits_key6.json: MISSING
  alice_sifted_bits_key7.json: MISSING
  bob_sifted_bits_key7.json: MISSING
  alice_sifted_bits_key8.json: MISSING
  bob_sifted_bits_key8.json: MISSING
  alice_sifted_bits_key9.json: MISSING
  bob_sifted_bits_key9.json: MISSING
  key_pairs_metadata.csv: False

=== remote (bob, site-1 slice) state ===
  key0 (alice file on bob node): present
  key1 (alice file on bo

In [2]:
import json

alice = slice_obj.get_node("alice")
recovered = []
for i in range(5):
    for remote_name in [f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json"]:
        local_path = RESULTS / remote_name
        stdout, _ = bob.execute(f"cat ~/qfabric/results/{remote_name}", quiet=True)
        try:
            data = json.loads(stdout)
        except json.JSONDecodeError:
            print(f"  {remote_name}: remote read failed, skipping")
            continue
        local_path.write_text(stdout)
        recovered.append((remote_name, len(data.get("alice_bits", data.get("bob_bits", [])))))
        print(f"  recovered {remote_name} ({recovered[-1][1]} bits)")

print(f"\nRecovered {len(recovered)} files.")

  recovered alice_sifted_bits_key0.json (3504 bits)
  recovered bob_sifted_bits_key0.json (3504 bits)
  recovered alice_sifted_bits_key1.json (3409 bits)
  recovered bob_sifted_bits_key1.json (3409 bits)
  recovered alice_sifted_bits_key2.json (3360 bits)
  recovered bob_sifted_bits_key2.json (3360 bits)
  recovered alice_sifted_bits_key3.json (3431 bits)
  recovered bob_sifted_bits_key3.json (3431 bits)
  recovered alice_sifted_bits_key4.json (3456 bits)
  recovered bob_sifted_bits_key4.json (3456 bits)

Recovered 10 files.


In [3]:
import pandas as pd

# From your original snapshot -- cross-check n_bits against what Step 1 recovered above
known_metadata = [
    {"index": 0, "m_sifted": 3894, "k_pe": 390, "n_bits": 3504, "qber": 0.01282051282051282},
    {"index": 1, "m_sifted": 3788, "k_pe": 379, "n_bits": 3409, "qber": 0.0158311345646438},
    {"index": 2, "m_sifted": 3734, "k_pe": 374, "n_bits": 3360, "qber": 0.00267379679144385},
    {"index": 3, "m_sifted": 3813, "k_pe": 382, "n_bits": 3431, "qber": 0.002617801047120419},
    {"index": 4, "m_sifted": 3841, "k_pe": 385, "n_bits": 3456, "qber": 0.005194805194805195},
]
for row in known_metadata:
    i = row["index"]
    row["alice_path"] = str(RESULTS / f"alice_sifted_bits_key{i}.json")
    row["bob_path"] = str(RESULTS / f"bob_sifted_bits_key{i}.json")
    # _raw files were never uploaded remotely (only the post-PE-split generation
    # files were), so these two are genuinely unrecoverable -- left blank rather
    # than pointing at something that doesn't exist.
    row["alice_raw_path"] = None
    row["bob_raw_path"] = None

pd.DataFrame(known_metadata).to_csv(str(RESULTS / "key_pairs_metadata.csv"), index=False)
print("wrote key_pairs_metadata.csv for keys 0-4")

wrote key_pairs_metadata.csv for keys 0-4


In [8]:
# --- Site-1 vs. site-2 CMH test, matched to your existing mock-vs-real methodology
# (same cmh_manual / stratify-by-prob pattern as notebooks/sdc_sweeps_all5keys.ipynb) ---
from scipy.stats import chi2

def cmh_manual(tables):
    num, den = 0.0, 0.0
    for t in tables:
        a, b, c, d = t[0, 0], t[0, 1], t[1, 0], t[1, 1]
        n = a + b + c + d
        if n < 2:
            continue
        row1, row2, col1, col2 = a + b, c + d, a + c, b + d
        E_a = row1 * col1 / n
        var_a = row1 * row2 * col1 * col2 / (n**2 * (n - 1))
        num += (a - E_a)
        den += var_a
    if den == 0:
        return np.nan, np.nan
    stat = num**2 / den
    return stat, 1 - chi2.cdf(stat, 1)


def build_tables_silent_both(site1_df, site2_df, fault_type):
    """One 2x2 table per firing-probability stratum, restricting BOTH sides
    to silent (verification-passed) mismatches, same as your mock-vs-real test."""
    tables = []
    for p in sorted(site2_df["prob"].unique()):
        s1 = site1_df[(site1_df["prob"] == p)]
        s2 = site2_df[(site2_df["prob"] == p) & (site2_df["fault_type"] == fault_type)]
        if len(s1) == 0 or len(s2) == 0:
            continue
        s1_mis = ((~s1["keys_match"].astype(bool)) & (s1["verification_passed"] == True)).sum()
        s2_mis = ((~s2["keys_match"].astype(bool)) & (s2["verification_passed"] == True)).sum()
        tables.append(np.array([[s1_mis, len(s1) - s1_mis], [s2_mis, len(s2) - s2_mis]]))
    return tables

site2 = pd.read_csv(str(CSV_SITE2))

# Your existing site-1 (NCSA-EDUKY) real-channel CSVs from the original submission:
site1_toeplitz = pd.read_csv(str(RESULTS / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv"))
site1_finalkey = pd.read_csv(str(RESULTS / "sdc_realchannel_finalkey_allkeys_asymptotic.csv"))

print("=== Site 1 (NCSA<->EDUKY) vs. Site 2 (NCSA<->UTAH), pooled CMH test ===\n")
for name, site1_df, ftype in [("toeplitz", site1_toeplitz, "toeplitz"),
                               ("final_key", site1_finalkey, "final_key")]:
    tables = build_tables_silent_both(site1_df, site2, ftype)
    stat, p = cmh_manual(tables)
    n1 = sum(t[0].sum() for t in tables)
    n2 = sum(t[1].sum() for t in tables)
    print(f"{name}: n_site1={n1}, n_site2={n2}, CMH stat={stat:.3f}, p={p:.4f} "
          f"-- {'SIGNIFICANT (site matters)' if p < 0.05 else 'not significant (consistent with Section 4.3)'}")


=== Site 1 (NCSA<->EDUKY) vs. Site 2 (NCSA<->UTAH), pooled CMH test ===

toeplitz: n_site1=1200, n_site2=580, CMH stat=35.533, p=0.0000 -- SIGNIFICANT (site matters)
final_key: n_site1=1200, n_site2=560, CMH stat=82.759, p=0.0000 -- SIGNIFICANT (site matters)


## Section C — 5 more real keys, 5 → 10 (Feedback #9)

Extends `collect_key_pairs` with a `start_index`, so the new keys land at indices 5-9
without touching your existing `key_pairs_metadata.csv` (indices 0-4) or the `key{i}` JSON
files that Sections 4-8 of the paper already depend on. Then recomputes Table 2
(finite-key vs. asymptotic length) for all 10 keys, so `ell_finite`'s sensitivity to any
single key (key 1 at `ell=1` had to be excluded as degenerate) is easier to characterize.

In [4]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade.key import key_from_sifted_json
from qne.cascade.parameter_estimation import split_pe_and_generation
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length

RESULTS = PROJECT_DIR / "results"

SLICE_NAME = 'qfabric-bb84-2'  # site-1 slice, same as Section A / the original 5 keys
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


In [6]:
def collect_key_pairs_from(deploy, slice_obj, alice, bob, bob_ip, project_dir,
                           start_index, n_keys, scenario_path="validation/scenarios/fabric_1km.yml",
                           pe_sample_fraction=0.1, pe_seed_base=100000):
    """Same body as qne.cascade.sweep_utils.collect_key_pairs, but starts at
    start_index and APPENDS to key_pairs_metadata.csv instead of overwriting it --
    so your existing 5 keys (indices 0-4) are preserved."""
    alice_mac = alice.get_interface(network_name="net_alice_switch").get_mac()
    bob_mac = bob.get_interface(network_name="net_switch_bob").get_mac()
    sw_alice_mac = slice_obj.get_node("switch").get_interface(network_name="net_alice_switch").get_mac()

    key_pairs = []
    for i in range(start_index, start_index + n_keys):
        print(f"\n=== Collecting key pair {i} ===")
        deploy.run_bb84(slice_obj, scenario_path, alice_mac, bob_mac,
                          sw_alice_mac=sw_alice_mac, bob_data_ip=bob_ip)

        stdout, _ = alice.execute("cat ~/qfabric/results/alice_sifted_bits.json", quiet=True)
        alice_raw_path = project_dir / "results" / f"alice_sifted_bits_key{i}_raw.json"
        alice_raw_path.write_text(stdout)

        stdout, _ = bob.execute("cat ~/qfabric/results/bob_sifted_bits.json", quiet=True)
        bob_raw_path = project_dir / "results" / f"bob_sifted_bits_key{i}_raw.json"
        bob_raw_path.write_text(stdout)

        alice_key_full, alice_idx_full = key_from_sifted_json(str(alice_raw_path), "alice_bits")
        bob_key_full, bob_idx_full = key_from_sifted_json(str(bob_raw_path), "bob_bits")

        if alice_idx_full != bob_idx_full:
            print(f"  key pair {i}: index mismatch -- skipping (raw files saved for inspection)")
            continue

        try:
            split = split_pe_and_generation(alice_key_full, bob_key_full,
                                             sample_fraction=pe_sample_fraction, seed=pe_seed_base + i)
        except ValueError as e:
            print(f"  key pair {i}: PE split failed ({e}) -- skipping")
            continue

        m, k, n = split["m"], split["k"], split["n"]
        qber_i = split["qber"]
        alice_key_i, bob_key_i = split["alice_gen"], split["bob_gen"]
        print(f"  key pair {i}: m={m} sifted, k={k} PE sample, n={n} generation bits, qber={qber_i:.4f}")

        alice_path = project_dir / "results" / f"alice_sifted_bits_key{i}.json"
        bob_path = project_dir / "results" / f"bob_sifted_bits_key{i}.json"
        gen_indices = split["gen_indices"]
        alice_matching_indices_gen = [alice_idx_full[idx] for idx in gen_indices]

        alice_path.write_text(json.dumps({"alice_bits": alice_key_i.bits.tolist(),
                                           "matching_indices": alice_matching_indices_gen}))
        bob_path.write_text(json.dumps({"bob_bits": bob_key_i.bits.tolist(),
                                         "matching_indices": alice_matching_indices_gen}))

        key_pairs.append({"index": i, "m_sifted": m, "k_pe": k, "n_bits": n, "qber": qber_i,
                           "alice_path": str(alice_path), "bob_path": str(bob_path),
                           "alice_raw_path": str(alice_raw_path), "bob_raw_path": str(bob_raw_path)})

        bob.upload_file(str(alice_path), f"qfabric/results/alice_sifted_bits_key{i}.json")
        bob.upload_file(str(bob_path), f"qfabric/results/bob_sifted_bits_key{i}.json")

    return pd.DataFrame(key_pairs)


existing_meta = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))
new_keys_df = collect_key_pairs_from(deploy, slice_obj, alice, bob, bob_ip, PROJECT_DIR,
                                      start_index=5, n_keys=5)
combined_meta = pd.concat([existing_meta, new_keys_df], ignore_index=True)
combined_meta.to_csv(str(RESULTS / "key_pairs_metadata.csv"), index=False)
print(f"\nkey_pairs_metadata.csv now has {len(combined_meta)} keys (indices "
      f"{combined_meta['index'].min()}-{combined_meta['index'].max()})")


FileNotFoundError: [Errno 2] No such file or directory: '/home/fabric/work/qkd-dependability/results/key_pairs_metadata.csv'

In [14]:
# --- Recompute Table 2 (finite-key vs. asymptotic length) for all 10 keys ---
rows = []
for _, row in combined.iterrows():
    i, n, k, qber = int(row["index"]), int(row["n_bits"]), int(row["k_pe"]), float(row["qber"])
    ell_asymptotic = asymptotic_key_length(n, qber)
    try:
        ell_finite, r, t, nu = finite_key_output_length(n, k, qber)
    except ValueError as e:
        ell_finite = None
        print(f"key {i}: finite-key optimization failed ({e})")
    reduction = (1 - ell_finite / ell_asymptotic) * 100 if ell_finite and ell_asymptotic else None
    rows.append({"key": i, "n_bits": n, "qber": qber,
                 "ell_asymptotic": ell_asymptotic, "ell_finite": ell_finite, "reduction_pct": reduction})

table2_extended = pd.DataFrame(rows)
table2_extended.to_csv(str(RESULTS / "finitekey_vs_asymptotic_table2_10keys.csv"), index=False)
print(table2_extended.to_string(index=False))
print(f"\nOriginal 5-key mean reduction: {table2_extended.iloc[:5]['reduction_pct'].mean():.2f}%")
print(f"All-10-key mean reduction:     {table2_extended['reduction_pct'].mean():.2f}%")
print("(Exclude any key with an extreme ell_finite, e.g. <=1 bit, from the mean the same "
      "way key 1 was excluded in the original 5-key table.)")


NameError: name 'asymptotic_key_length' is not defined

In [13]:
import pandas as pd
combined = pd.DataFrame(known_metadata + recovered_5_to_9)
combined.to_csv(str(RESULTS / "key_pairs_metadata.csv"), index=False)
print(f"key_pairs_metadata.csv rebuilt with {len(combined)} keys")

key_pairs_metadata.csv rebuilt with 10 keys


In [3]:
import pandas as pd
site1_meta = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv")).iloc[:5]   # keys 0-4
site2_meta = pd.read_csv(str(RESULTS / "key_pairs_metadata_site2.csv"))
print("site 1 QBER:", sorted(site1_meta["qber"].tolist()))
print("site 2 QBER:", sorted(site2_meta["qber"].tolist()))

site 1 QBER: [0.0026178010471204, 0.0026737967914438, 0.0051948051948051, 0.0128205128205128, 0.0158311345646438]
site 2 QBER: [0.0102827763496143, 0.0105263157894736, 0.0153846153846153, 0.021108179419525, 0.0212765957446808]


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2

RESULTS = PROJECT_DIR / "results"

site1_toeplitz = pd.read_csv(str(RESULTS / "sdc_realchannel_toeplitz_allkeys_asymptotic.csv"))
site1_finalkey = pd.read_csv(str(RESULTS / "sdc_realchannel_finalkey_allkeys_asymptotic.csv"))
site2 = pd.read_csv(str(RESULTS / "sdc_site2_toeplitz_finalkey_doseresponse.csv"))

def cmh_manual(tables):
    num, den = 0.0, 0.0
    for t in tables:
        a, b, c, d = t[0, 0], t[0, 1], t[1, 0], t[1, 1]
        n = a + b + c + d
        if n < 2:
            continue
        row1, row2, col1, col2 = a + b, c + d, a + c, b + d
        E_a = row1 * col1 / n
        var_a = row1 * row2 * col1 * col2 / (n**2 * (n - 1))
        num += (a - E_a)
        den += var_a
    if den == 0:
        return np.nan, np.nan
    stat = num**2 / den
    return stat, 1 - chi2.cdf(stat, 1)


def add_ell_ratio(df):
    """The same quantity your model (Eq. 1) actually predicts P(D) from --
    ell/(ell+t_verify) -- computed per-trial from each row's OWN key's QBER,
    not from which site it came from."""
    df = df.copy()
    df["ell_ratio"] = df["secure_key_length"] / (df["secure_key_length"] + df["t_verify"])
    return df

site1_toeplitz = add_ell_ratio(site1_toeplitz)
site1_finalkey = add_ell_ratio(site1_finalkey)
site2 = add_ell_ratio(site2)

# Bin ell_ratio into quartiles computed over the POOLED range across both sites,
# so "same bin" means "the model predicts roughly the same P(D) regardless of
# which site the trial came from" -- this is what actually controls for the
# QBER confound, rather than requiring the raw QBER values themselves to match.
def build_tables_ell_controlled(site1_df, site2_df, fault_type, n_bins=4):
    s1 = site1_df  # site1_toeplitz/finalkey is already fault-type-specific
    s2 = site2_df[site2_df["fault_type"] == fault_type]

    pooled_ratio = pd.concat([s1["ell_ratio"], s2["ell_ratio"]])
    bin_edges = np.quantile(pooled_ratio, np.linspace(0, 1, n_bins + 1))
    bin_edges[0] -= 1e-9  # include the min value in the first bin

    tables, bin_report = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        for p in sorted(set(s1["prob"].unique()) & set(s2["prob"].unique())):
            s1_bin = s1[(s1["prob"] == p) & (s1["ell_ratio"] > lo) & (s1["ell_ratio"] <= hi)]
            s2_bin = s2[(s2["prob"] == p) & (s2["ell_ratio"] > lo) & (s2["ell_ratio"] <= hi)]
            if len(s1_bin) < 3 or len(s2_bin) < 3:
                continue  # too few trials in this (ell-bin, prob) cell to say anything
            s1_mis = ((~s1_bin["keys_match"].astype(bool)) & (s1_bin["verification_passed"] == True)).sum()
            s2_mis = ((~s2_bin["keys_match"].astype(bool)) & (s2_bin["verification_passed"] == True)).sum()
            tables.append(np.array([[s1_mis, len(s1_bin) - s1_mis], [s2_mis, len(s2_bin) - s2_mis]]))
            bin_report.append((round(lo, 3), round(hi, 3), p, len(s1_bin), len(s2_bin)))
    return tables, bin_report

print("=== Site 1 vs. Site 2, controlling for ell/(ell+t_verify) via stratification ===\n")
for name, site1_df, ftype in [("toeplitz", site1_toeplitz, "toeplitz"), ("final_key", site1_finalkey, "final_key")]:
    tables, bin_report = build_tables_ell_controlled(site1_df, site2, ftype)
    print(f"{name}: {len(tables)} usable (ell-bin, prob) strata with >=3 trials on both sides")
    for lo, hi, p, n1, n2 in bin_report:
        print(f"    ell_ratio in ({lo}, {hi}], prob={p}: n_site1={n1}, n_site2={n2}")
    if len(tables) == 0:
        print("    -> no usable strata at all; cannot say anything about a site effect from this data.\n")
        continue
    stat, p_val = cmh_manual(tables)
    n1_total = sum(t[0].sum() for t in tables)
    n2_total = sum(t[1].sum() for t in tables)
    print(f"    CMH stat={stat:.3f}, p={p_val:.4f} (n_site1={n1_total}, n_site2={n2_total}) "
          f"-- {'SIGNIFICANT even controlling for ell' if p_val < 0.05 else 'not significant once ell is controlled for'}\n")

In [3]:
site2 = pd.read_csv(str(RESULTS / "sdc_site2_toeplitz_finalkey_doseresponse.csv"))
print("columns:", site2.columns.tolist())
print()
print("row count:", len(site2))
print("error column value counts:")
print(site2["error"].value_counts(dropna=False) if "error" in site2.columns else "  (no 'error' column)")
print()
print("non_convergent value counts:")
print(site2["non_convergent"].value_counts(dropna=False) if "non_convergent" in site2.columns else "  (no 'non_convergent' column)")
print()
print("keys_match value counts:")
print(site2["keys_match"].value_counts(dropna=False) if "keys_match" in site2.columns else "  (no 'keys_match' column)")
print()
print("verification_passed value counts:")
print(site2["verification_passed"].value_counts(dropna=False) if "verification_passed" in site2.columns else "  (no 'verification_passed' column)")

columns: ['run', 'seed', 'error', 'raw_stdout', 'raw_stderr', 'keys_match', 'verification_passed', 'non_convergent', 'code_version', 'collection_timestamp', 'fault_type', 'prob', 'key_index', 'qber']

row count: 1140
error column value counts:
error
no bob output path found in stdout    1140
Name: count, dtype: int64

non_convergent value counts:
non_convergent
True    1140
Name: count, dtype: int64

keys_match value counts:
keys_match
NaN    1140
Name: count, dtype: int64

verification_passed value counts:
verification_passed
NaN    1140
Name: count, dtype: int64


In [4]:
print("=== raw_stdout ===")
print(site2["raw_stdout"].iloc[0])
print("\n=== raw_stderr ===")
print(site2["raw_stderr"].iloc[0])

=== raw_stdout ===
nan

=== raw_stderr ===
Traceback (most recent call last):
  File "/home/ubuntu/qfabric/scripts/bob_cascade_driver.py", line 25, in <module>
    from galois import GF2
ModuleNotFoundError: No module named 'galois'



In [5]:
# --- 1. Install the deps install_deps() actually missed, on both site-2 nodes ---
for node in [alice_2, bob_2]:
    _, stderr = node.execute(
        "cd ~/qfabric && source .venv/bin/activate && pip install --quiet -r requirements.txt",
        quiet=True,
    )
    check, _ = node.execute(
        "~/qfabric/.venv/bin/python3 -c 'from galois import GF2; import randextract; print(\"deps OK\")'",
        quiet=True,
    )
    print(f"{node.get_name()}: {check.strip() or ('FAILED: ' + stderr[-500:])}")
    if "deps OK" not in check:
        raise RuntimeError(f"{node.get_name()} still missing deps after install -- stopping before wasting FABRIC time on the sweep.")

print("\ndeps fixed on both nodes, safe to proceed")

NameError: name 'alice_2' is not defined

In [16]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS = PROJECT_DIR / "results"

def show_key_data(metadata_csv_name):
    meta = pd.read_csv(str(RESULTS / metadata_csv_name))
    print(meta.to_string(index=False))

    # If the raw bit files are present, also show the true (measured) QBER
    # alongside the PE-sample estimate -- no modeling, just direct comparison.
    rows = []
    for _, row in meta.iterrows():
        idx = int(row["index"])
        n_bits = int(row["n_bits"])
        q_hat = float(row["qber"])
        alice_path = RESULTS / Path(row["alice_path"]).name if "alice_path" in row and pd.notna(row["alice_path"]) else None
        bob_path = RESULTS / Path(row["bob_path"]).name if "bob_path" in row and pd.notna(row["bob_path"]) else None
        true_qber = None
        if alice_path and bob_path and alice_path.exists() and bob_path.exists():
            with open(alice_path) as f:
                alice_bits = np.array(json.load(f)["alice_bits"], dtype=np.uint8)
            with open(bob_path) as f:
                bob_bits = np.array(json.load(f)["bob_bits"], dtype=np.uint8)
            true_qber = int(np.sum(alice_bits != bob_bits)) / n_bits
        rows.append({"index": idx, "n_bits": n_bits, "q_hat": q_hat, "true_qber": true_qber})

    print("\n--- q_hat vs. true qber (measured directly, no model) ---")
    print(pd.DataFrame(rows).to_string(index=False))

show_key_data("key_pairs_metadata.csv")       # or "key_pairs_metadata_site2.csv" for the Tokyo/NCSA keys

 index  m_sifted  k_pe  n_bits     qber                                                              alice_path                                                              bob_path  alice_raw_path  bob_raw_path
     0      3894   390    3504 0.012821 /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key0.json /home/fabric/work/qkd-dependability/results/bob_sifted_bits_key0.json             NaN           NaN
     1      3788   379    3409 0.015831 /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key1.json /home/fabric/work/qkd-dependability/results/bob_sifted_bits_key1.json             NaN           NaN
     2      3734   374    3360 0.002674 /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key2.json /home/fabric/work/qkd-dependability/results/bob_sifted_bits_key2.json             NaN           NaN
     3      3813   382    3431 0.002618 /home/fabric/work/qkd-dependability/results/alice_sifted_bits_key3.json /home/fabric/work/qkd-dependability/resu

In [17]:
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length

meta = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))
rows = []
for _, row in meta.iterrows():
    i, n, k, qber = int(row["index"]), int(row["n_bits"]), int(row["k_pe"]), float(row["qber"])
    ell_asymptotic = asymptotic_key_length(n, qber)
    try:
        ell_finite, r, t, nu = finite_key_output_length(n, k, qber)
    except ValueError as e:
        ell_finite = None
        print(f"key {i}: finite-key optimization failed ({e})")
    reduction = (1 - ell_finite / ell_asymptotic) * 100 if ell_finite and ell_asymptotic else None
    rows.append({"key": i, "n_bits": n, "qber": qber,
                 "ell_asymptotic": ell_asymptotic, "ell_finite": ell_finite, "reduction_pct": reduction})

table2 = pd.DataFrame(rows)
table2.to_csv(str(RESULTS / "finitekey_vs_asymptotic_table2.csv"), index=False)
print(table2.to_string(index=False))

degenerate = table2[table2["ell_finite"] <= 1]
clean = table2[~table2.index.isin(degenerate.index)]
print(f"\nmean reduction (excluding {len(degenerate)} degenerate ell<=1 key(s)): {clean['reduction_pct'].mean():.2f}%")
if len(degenerate):
    print(f"degenerate keys (ell_finite<=1): {degenerate['key'].tolist()}")

 key  n_bits     qber  ell_asymptotic  ell_finite  reduction_pct
   0    3504 0.012821            2810          23      99.181495
   1    3409 0.015831            2608           1      99.961656
   2    3360 0.002674            3180         337      89.402516
   3    3431 0.002618            3251         363      88.834205
   4    3456 0.005195            3131         269      91.408496
   5    3455 0.010417            2877          87      96.976017
   6    3446 0.023499            2338           1      99.957228
   7    3388 0.010610            2813          66      97.653750
   8    3430 0.010471            2854          81      97.161878
   9    3351 0.013405            2663           1      99.962448

mean reduction (excluding 3 degenerate ell<=1 key(s)): 94.37%
degenerate keys (ell_finite<=1): [1, 6, 9]


In [25]:
import sys, json
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
RESULTS = PROJECT_DIR / "results"

import deploy_fabric as deploy
fablib = deploy.get_fablib()

slice_obj_2 = fablib.get_slice(name='qfabric-bb84-site2')  # adjust if you named it differently
alice_2 = slice_obj_2.get_node("alice")
bob_2 = slice_obj_2.get_node("bob")
bob_ip_2 = "10.10.1.2"  # fill in from setup_dataplane_ips output, or derive below if unknown
if bob_ip_2 is None:
    stdout, _ = bob_2.execute("ip -4 addr show | grep inet", quiet=True)
    print(stdout)  # pick the dataplane-side address (matches your 10.10.x.x convention)

key_pairs_df_2 = pd.read_csv(str(RESULTS / "key_pairs_metadata_site2.csv"))
print(key_pairs_df_2)

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
   index  m_sifted  k_pe  n_bits      qber  \
0      0      3855   386    3469  0.007772   
1      1      3791   380    3411  0.002632   
2      2      3894   390    3504  0.023077   
3      3      3711   372    3339  0.016129   
4      4      3882   389    3493  0.005141   

                                          alice_path  \
0  /home/fabric/work/qkd-dependability/results/al...   
1  /home/fabric/work/qkd-dependability/results/al...   
2  /home/fabric/work/qkd-dependability/results/al...   
3  /home/fabric/work/qkd-dependability/results/al...   
4  /home/fabric/work/qkd-dependability/results/al...   

                                            bob_path  \
0  /home/fabric/work/qkd-dependability/results/bo...   
1  /home/fabric/work/qkd-dependability/results/bo...   
2  /home/fabric/work/qkd-dependability/results/bo...   
3  /home/fabric/work/qkd-dependability/results/bo...   
4  /home/fabric/work/qkd-dependabi

In [20]:
for node in [alice_2, bob_2]:
    _, stderr = node.execute(
        "cd ~/qfabric && source .venv/bin/activate && pip install --quiet -r requirements.txt",
        quiet=True,
    )
    check, _ = node.execute(
        "~/qfabric/.venv/bin/python3 -c 'from galois import GF2; import randextract; print(\"deps OK\")'",
        quiet=True,
    )
    print(f"{node.get_name()}: {check.strip() or ('FAILED: ' + stderr[-500:])}")
    if "deps OK" not in check:
        raise RuntimeError(f"{node.get_name()} still missing deps after install -- paste the stderr above.")

print("\ndeps fixed on both nodes, safe to proceed to Step 3")

alice: deps OK
bob: deps OK

deps fixed on both nodes, safe to proceed to Step 3


In [ ]:
from qne.cascade.sweep_utils import run_real_channel_trial

PROBS = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
N_RUNS_PER_PROB = 20  # raise once this runs cleanly end to end
CSV_SITE2 = RESULTS / "sdc_site2_toeplitz_finalkey_doseresponse.csv"

all_rows = []
first_trial_checked = False

for key_idx in range(len(key_pairs_df_2)):
    meta_row = key_pairs_df_2.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])
    a_remote = f"results/alice_sifted_bits_key{key_idx}_site2.json"
    b_remote = f"results/bob_sifted_bits_key{key_idx}_site2.json"

    # Confirm the per-key files are actually on the node before running trials
    check, _ = bob_2.execute(f"test -f ~/qfabric/{a_remote} && test -f ~/qfabric/{b_remote} && echo present || echo MISSING", quiet=True)
    if "present" not in check:
        a_local = RESULTS / f"alice_sifted_bits_key{key_idx}_site2.json"
        b_local = RESULTS / f"bob_sifted_bits_key{key_idx}_site2.json"
        alice_2.upload_file(str(a_local), f"qfabric/{a_remote}")
        bob_2.upload_file(str(a_local), f"qfabric/{a_remote}")
        bob_2.upload_file(str(b_local), f"qfabric/{b_remote}")
        print(f"  key{key_idx}: re-uploaded (was missing on node)")

    print(f"\n=== site2 key{key_idx}: qber={qber_i:.4f} ===")

    for prob in PROBS:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}),
                                    ("final_key", {"final_key_prob": prob})]:
            for run in range(N_RUNS_PER_PROB):
                seed = 42 + run
                result = run_real_channel_trial(bob_2, alice_2, bob_ip_2, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic",
                                                   bob_key_json=b_remote, alice_key_json=a_remote,
                                                   **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)

                if not first_trial_checked:
                    first_trial_checked = True
                    if result.get("error"):
                        raise RuntimeError(f"First trial failed: {result.get('error')!r}. "
                                            f"raw_stderr tail: {str(result.get('raw_stderr'))[-500:]}")
                    print(f"  first trial OK")

            pd.DataFrame(all_rows).to_csv(str(CSV_SITE2), index=False)

print(f"\nSaved {len(all_rows)} rows -> {CSV_SITE2}")


=== site2 key0: qber=0.0078 ===
  first trial OK
